## PACOTES ##

In [ ]:

import io
import re
import html
import base64
import time
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import confusion_matrix


## CÓDIGO ##

In [ ]:

AZUL_NAO_FRAUDE = "#2563eb"
AZUL_NAO_FRAUDE_BORDA = "#1e3a8a"
AZUL_NAO_FRAUDE_CLARO = "#60a5fa"
AMARELO_FRAUDE = "#facc15"
PRETO_BORDA = "#111827"

def sanitizar_nome(nome):
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(nome))


def formatar_count(valor):
    return f"{int(valor):,}".replace(",", ".")


def formatar_float_html(valor, casas=6):
    if pd.isna(valor):
        return "-"
    try:
        return f"{float(valor):.{casas}f}".replace(".", ",")
    except Exception:
        return str(valor)


def formatar_float_latex(valor, casas=6):
    if pd.isna(valor):
        return "-"
    try:
        return f"{float(valor):.{casas}f}"
    except Exception:
        return str(valor)


def fig_to_base64(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=150, bbox_inches="tight")
    buffer.seek(0)
    return base64.b64encode(buffer.read()).decode("utf-8")


def salvar_figura(fig, caminho_sem_extensao):
    caminho_sem_extensao = Path(caminho_sem_extensao)
    fig.savefig(caminho_sem_extensao.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(caminho_sem_extensao.with_suffix(".png"), dpi=300, bbox_inches="tight")


def gerar_tabela_html(df, table_id, classe="data-table"):
    if df is None or df.empty:
        return "<p>Nenhum dado disponível.</p>"

    html_tabela = f'<table id="{html.escape(str(table_id))}" class="{classe}">\n'
    html_tabela += "<thead><tr>"

    for col in df.columns:
        html_tabela += f"<th>{html.escape(str(col))}</th>"

    html_tabela += "</tr></thead><tbody>\n"

    for _, row in df.iterrows():
        html_tabela += "<tr>"
        for valor in row:
            html_tabela += f"<td>{html.escape(str(valor))}</td>"
        html_tabela += "</tr>\n"

    html_tabela += "</tbody></table>"
    return html_tabela


def gerar_secao_tabela(titulo, tabela_html, table_id, nome_csv):
    return f"""
    <section class="plot-card">
        <div class="section-header">
            <h2>{html.escape(str(titulo))}</h2>
            <button class="download-btn" onclick="baixarTabelaCSV('{html.escape(str(table_id))}', '{html.escape(str(nome_csv))}')">
                Baixar CSV
            </button>
        </div>
        <div class="table-wrapper">
            {tabela_html}
        </div>
    </section>
    """


def preparar_df_latex(df):
    df_latex = df.copy()

    for col in df_latex.columns:
        if pd.api.types.is_float_dtype(df_latex[col]):
            df_latex[col] = df_latex[col].apply(lambda x: formatar_float_latex(x, 6))
        elif pd.api.types.is_integer_dtype(df_latex[col]):
            df_latex[col] = df_latex[col].apply(lambda x: int(x) if not pd.isna(x) else x)

    return df_latex


def salvar_tabela_latex(df, caminho, caption, label, longtable=False):
    caminho = Path(caminho)

    if df is None or df.empty:
        caminho.write_text("% Tabela vazia.\n", encoding="utf-8")
        return

    tex = preparar_df_latex(df).to_latex(
        index=False,
        escape=True,
        longtable=longtable,
        caption=caption,
        label=label
    )

    caminho.write_text(tex, encoding="utf-8")


def exportar_tabelas_latex(pasta_latex, tabela_metricas, tabela_matrizes):
    pasta_latex = Path(pasta_latex)
    pasta_latex.mkdir(parents=True, exist_ok=True)

    salvar_tabela_latex(
        tabela_metricas,
        pasta_latex / "tabela_metricas_ranks.tex",
        caption="Métricas dos melhores rankings do experimento t-SNE 2D.",
        label="tab:metricas-ranks-2x2",
        longtable=False
    )

    salvar_tabela_latex(
        tabela_matrizes,
        pasta_latex / "tabela_matrizes_confusao_ranks.tex",
        caption="Matrizes de confusão dos melhores rankings do experimento t-SNE 2D.",
        label="tab:matrizes-confusao-ranks-2x2",
        longtable=True
    )

    comandos_latex = r"""

% Pacotes recomendados:
% \usepackage{graphicx}
% \usepackage{float}
% \usepackage{booktabs}
% \usepackage{longtable}
% \usepackage{pdflscape}

\begin{table}[H]
\centering
\caption{Métricas dos melhores rankings do experimento t-SNE 2D.}
\label{tab:metricas-ranks-2x2-main}
\input{2x2_tsne_visu_scores/tabela_metricas_ranks.tex}
\end{table}

\begin{landscape}
\small
\input{2x2_tsne_visu_scores/tabela_matrizes_confusao_ranks.tex}
\end{landscape}

\begin{figure}[H]
\centering
\includegraphics[width=\textwidth]{2x2_tsne_visu_scores/rank_1_tsne2d_perplexity_EXEMPLO_dispersao_2d.pdf}
\caption{Dispersão 2D do t-SNE selecionado no Rank 1.}
\label{fig:rank1-2x2-dispersao}
\end{figure}
"""

    (pasta_latex / "comandos_latex_exemplo.tex").write_text(comandos_latex, encoding="utf-8")


def gerar_matriz_confusao(y_real, probabilidades, threshold):
    y_pred = (probabilidades >= threshold).astype(int)
    return confusion_matrix(y_real, y_pred, labels=[0, 1])


def preparar_valores_matriz(cm):
    tn, fp, fn, tp = cm.ravel()

    total_fraudes = fn + tp
    total_nao_fraudes = tn + fp

    fn_pct = fn / total_fraudes * 100 if total_fraudes != 0 else 0
    tp_pct = tp / total_fraudes * 100 if total_fraudes != 0 else 0

    tn_pct = tn / total_nao_fraudes * 100 if total_nao_fraudes != 0 else 0
    fp_pct = fp / total_nao_fraudes * 100 if total_nao_fraudes != 0 else 0

    return {
        "fn": {"pct": fn_pct, "count": int(fn), "qualidade": 100 - fn_pct},
        "tp": {"pct": tp_pct, "count": int(tp), "qualidade": tp_pct},
        "tn": {"pct": tn_pct, "count": int(tn), "qualidade": tn_pct},
        "fp": {"pct": fp_pct, "count": int(fp), "qualidade": 100 - fp_pct},
        "raw": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
    }


def preparar_valores_matriz_ideal(y_real):
    y_real_array = np.asarray(y_real).astype(int)

    total_fraudes = int(np.sum(y_real_array == 1))
    total_nao_fraudes = int(np.sum(y_real_array == 0))

    return {
        "fn": {"pct": 0.0, "count": 0, "qualidade": 100.0},
        "tp": {"pct": 100.0, "count": total_fraudes, "qualidade": 100.0},
        "tn": {"pct": 100.0, "count": total_nao_fraudes, "qualidade": 100.0},
        "fp": {"pct": 0.0, "count": 0, "qualidade": 100.0},
        "raw": {"tn": total_nao_fraudes, "fp": 0, "fn": 0, "tp": total_fraudes}
    }


def cor_por_qualidade(q):
    if q >= 95:
        return "cell q95"
    elif q >= 85:
        return "cell q85"
    elif q >= 70:
        return "cell q70"
    elif q >= 50:
        return "cell q50"
    elif q >= 30:
        return "cell q30"
    else:
        return "cell q10"


def gerar_html_matriz(titulo, valores, matriz_ideal=False, imagem_base64=None, nome_imagem=None):
    if matriz_ideal:
        desc_fn = "Erro ideal: nenhuma fraude perdida"
        desc_tp = "Acerto ideal: fraudes detectadas"
        desc_tn = "Acerto ideal: não fraudes corretas"
        desc_fp = "Erro ideal: nenhum falso alerta"
    else:
        desc_fn = "Erro: fraude perdida"
        desc_tp = "Acerto: fraude detectada"
        desc_tn = "Acerto: não fraude"
        desc_fp = "Erro: falso alerta"

    botao = ""

    if imagem_base64 is not None and nome_imagem is not None:
        botao = f"""
        <div class="section-actions-only">
            <a class="download-btn link-btn" href="data:image/png;base64,{imagem_base64}" download="{html.escape(nome_imagem)}">
                Baixar PNG
            </a>
        </div>
        """

    return f"""
    <section class="matrix-card">
        <h2>{html.escape(str(titulo))}</h2>
        {botao}
        <div class="matrix-area">
            <div class="matrix-wrapper">
                <div class="corner"></div>
                <div class="x-label">Pred Não Fraude</div>
                <div class="x-label">Pred Fraude</div>

                <div class="y-label">Real Fraude</div>

                <div class="{cor_por_qualidade(valores['fn']['qualidade'])}">
                    <div class="pct">{valores['fn']['pct']:.2f}%</div>
                    <div class="count">({formatar_count(valores['fn']['count'])})</div>
                    <div class="cell-desc">{desc_fn}</div>
                </div>

                <div class="{cor_por_qualidade(valores['tp']['qualidade'])}">
                    <div class="pct">{valores['tp']['pct']:.2f}%</div>
                    <div class="count">({formatar_count(valores['tp']['count'])})</div>
                    <div class="cell-desc">{desc_tp}</div>
                </div>

                <div class="y-label">Real Não Fraude</div>

                <div class="{cor_por_qualidade(valores['tn']['qualidade'])}">
                    <div class="pct">{valores['tn']['pct']:.2f}%</div>
                    <div class="count">({formatar_count(valores['tn']['count'])})</div>
                    <div class="cell-desc">{desc_tn}</div>
                </div>

                <div class="{cor_por_qualidade(valores['fp']['qualidade'])}">
                    <div class="pct">{valores['fp']['pct']:.2f}%</div>
                    <div class="count">({formatar_count(valores['fp']['count'])})</div>
                    <div class="cell-desc">{desc_fp}</div>
                </div>
            </div>

            <div class="legend">
                <div class="legend-title">Qualidade</div>
                <div class="colorbar"></div>
                <div class="legend-label-top">Melhor</div>
                <div class="legend-label-bottom">Pior</div>
            </div>
        </div>
    </section>
    """


def gerar_fig_matriz_confusao(titulo, valores):
    matriz_pct = np.array([
        [valores["fn"]["pct"], valores["tp"]["pct"]],
        [valores["tn"]["pct"], valores["fp"]["pct"]]
    ])

    matriz_count = np.array([
        [valores["fn"]["count"], valores["tp"]["count"]],
        [valores["tn"]["count"], valores["fp"]["count"]]
    ])

    fig, ax = plt.subplots(figsize=(8.5, 6.2))

    im = ax.imshow(matriz_pct, vmin=0, vmax=100, cmap="Blues")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Percentual por classe real (%)", fontweight="bold")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(["Pred Não Fraude", "Pred Fraude"], fontweight="bold")
    ax.set_yticklabels(["Real Fraude", "Real Não Fraude"], fontweight="bold")

    ax.set_title(titulo, fontsize=14, fontweight="bold", pad=14)

    textos = [
        ["Fraude perdida", "Fraude detectada"],
        ["Não fraude correta", "Falso alerta"]
    ]

    for i in range(2):
        for j in range(2):
            ax.text(
                j,
                i,
                f"{matriz_pct[i, j]:.2f}%\n({formatar_count(matriz_count[i, j])})\n{textos[i][j]}",
                ha="center",
                va="center",
                color="black",
                fontweight="bold",
                fontsize=10
            )

    plt.tight_layout()
    return fig

def gerar_fig_dispersao_2d(df_plot, feature_1, feature_2, target_name):
    dados_nao_fraude = df_plot.loc[df_plot[target_name] == 0, [feature_1, feature_2]].dropna()
    dados_fraude = df_plot.loc[df_plot[target_name] == 1, [feature_1, feature_2]].dropna()

    fig, ax = plt.subplots(figsize=(11, 7))

    ax.scatter(
        dados_nao_fraude[feature_1],
        dados_nao_fraude[feature_2],
        s=6,
        alpha=0.08,
        color=AZUL_NAO_FRAUDE,
        rasterized=True
    )

    ax.scatter(
        dados_fraude[feature_1],
        dados_fraude[feature_2],
        s=30,
        alpha=0.90,
        color=AMARELO_FRAUDE,
        edgecolors=PRETO_BORDA,
        linewidths=0.25,
        rasterized=True
    )

    ax.set_title(f"Dispersão 2D por Classe Real - {feature_1} x {feature_2}", fontsize=14, fontweight="bold")
    ax.set_xlabel(feature_1, fontsize=14, fontweight="bold")
    ax.set_ylabel(feature_2, fontsize=14, fontweight="bold")
    ax.grid(alpha=0.25)

    legenda_nao_fraude = plt.Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        markersize=8,
        markerfacecolor=AZUL_NAO_FRAUDE,
        markeredgecolor=AZUL_NAO_FRAUDE_BORDA,
        label=f"Não Fraude ({len(dados_nao_fraude):,})".replace(",", ".")
    )

    legenda_fraude = plt.Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        markersize=8,
        markerfacecolor=AMARELO_FRAUDE,
        markeredgecolor=PRETO_BORDA,
        label=f"Fraude ({len(dados_fraude):,})".replace(",", ".")
    )

    ax.legend(handles=[legenda_nao_fraude, legenda_fraude], title="Classe Real", loc="best", frameon=True)

    plt.tight_layout()
    return fig


def gerar_fig_boxplots_2d(df_plot, feature_1, feature_2, target_name):
    features = [feature_1, feature_2]
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    cores = [AZUL_NAO_FRAUDE, AMARELO_FRAUDE]

    for ax, feature in zip(axes, features):
        dados_nao_fraude = df_plot.loc[df_plot[target_name] == 0, feature].dropna()
        dados_fraude = df_plot.loc[df_plot[target_name] == 1, feature].dropna()

        box = ax.boxplot(
            [dados_nao_fraude, dados_fraude],
            labels=["Não Fraude", "Fraude"],
            patch_artist=True,
            showfliers=True
        )

        for patch, cor in zip(box["boxes"], cores):
            patch.set_facecolor(cor)
            patch.set_alpha(0.70)
            patch.set_linewidth(2)

        for median in box["medians"]:
            median.set_color(PRETO_BORDA)
            median.set_linewidth(2.2)

        for whisker in box["whiskers"]:
            whisker.set_color("#334155")
            whisker.set_linewidth(1.6)

        for cap in box["caps"]:
            cap.set_color("#334155")
            cap.set_linewidth(1.6)

        for flier in box["fliers"]:
            flier.set_marker("o")
            flier.set_markerfacecolor("#64748b")
            flier.set_markeredgecolor("#64748b")
            flier.set_alpha(0.22)
            flier.set_markersize(2.5)

        ax.set_title(f"Boxplot - {feature}", fontsize=14, fontweight="bold", pad=12)
        ax.set_ylabel(feature, fontsize=13, fontweight="bold")
        ax.grid(axis="y", alpha=0.25)

    legenda_nao_fraude = plt.Line2D([0], [0], color=AZUL_NAO_FRAUDE, lw=8, label="Não Fraude")
    legenda_fraude = plt.Line2D([0], [0], color=AMARELO_FRAUDE, lw=8, label="Fraude")

    fig.legend(
        handles=[legenda_nao_fraude, legenda_fraude],
        title="Classe Real",
        loc="upper center",
        ncol=2,
        frameon=True,
        bbox_to_anchor=(0.5, 1.04)
    )

    plt.tight_layout()
    return fig


def gerar_fig_spearman(df_plot, feature_1, feature_2, target_name):
    dados_corr = df_plot[[feature_1, feature_2, target_name]].copy()
    dados_corr = dados_corr.rename(columns={target_name: "Fraude"})
    corr = dados_corr.corr(method="spearman")

    fig, ax = plt.subplots(figsize=(7.5, 6.2))

    im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Correlação de Spearman", fontsize=11, fontweight="bold")

    labels = corr.columns.tolist()

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))

    ax.set_xticklabels(labels, fontsize=12, fontweight="bold", rotation=35, ha="right")
    ax.set_yticklabels(labels, fontsize=12, fontweight="bold")

    ax.set_title(f"Spearman - {feature_1}, {feature_2} e Fraude", fontsize=13, fontweight="bold")

    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(
                j,
                i,
                f"{corr.values[i, j]:.3f}",
                ha="center",
                va="center",
                color="black",
                fontsize=13,
                fontweight="bold"
            )

    plt.tight_layout()
    return fig


def gerar_fig_responsabilidades_2d(
    df_plot,
    feature_1,
    feature_2,
    target_name,
    scaler,
    gmm,
    cluster_fraude,
    threshold,
    titulo
):
    y_real_array = df_plot[target_name].astype(int).to_numpy()
    mask_nao_fraude = y_real_array == 0
    mask_fraude = y_real_array == 1

    x_min = df_plot[feature_1].min()
    x_max = df_plot[feature_1].max()
    y_min = df_plot[feature_2].min()
    y_max = df_plot[feature_2].max()

    margem_x = 0.05 * (x_max - x_min)
    margem_y = 0.05 * (y_max - y_min)

    x_grid = np.linspace(x_min - margem_x, x_max + margem_x, 350)
    y_grid = np.linspace(y_min - margem_y, y_max + margem_y, 350)

    xx, yy = np.meshgrid(x_grid, y_grid)

    grid_original = np.c_[xx.ravel(), yy.ravel()]
    grid_original_df = pd.DataFrame(grid_original, columns=[feature_1, feature_2])
    grid_scaled = scaler.transform(grid_original_df)

    grid_prob = gmm.predict_proba(grid_scaled)[:, cluster_fraude]
    zz = grid_prob.reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(11, 7))

    fundo = ax.contourf(
        xx,
        yy,
        zz,
        levels=np.linspace(0, 1, 21),
        cmap="coolwarm",
        alpha=0.34,
        vmin=0,
        vmax=1
    )

    try:
        ax.contour(
            xx,
            yy,
            zz,
            levels=[0.25, 0.50, 0.75],
            colors="#475569",
            linewidths=0.75,
            linestyles=":",
            alpha=0.65
        )
    except Exception:
        pass

    try:
        ax.contour(
            xx,
            yy,
            zz,
            levels=[threshold],
            colors="#020617",
            linewidths=2.4,
            linestyles="--"
        )
    except Exception:
        pass

    ax.scatter(
        df_plot.loc[mask_nao_fraude, feature_1],
        df_plot.loc[mask_nao_fraude, feature_2],
        s=5,
        alpha=0.055,
        color=AZUL_NAO_FRAUDE,
        rasterized=True
    )

    ax.scatter(
        df_plot.loc[mask_fraude, feature_1],
        df_plot.loc[mask_fraude, feature_2],
        s=38,
        alpha=0.92,
        color=AMARELO_FRAUDE,
        edgecolors=PRETO_BORDA,
        linewidths=0.35,
        rasterized=True
    )

    cbar = plt.colorbar(fundo, ax=ax, fraction=0.046, pad=0.035)
    cbar.set_label("Responsabilidade GMM para fraude", fontsize=11, fontweight="bold")
    cbar.set_ticks([0, 0.25, 0.50, 0.75, 1.00])
    cbar.set_ticklabels(["0.00\nBaixa", "0.25", "0.50", "0.75", "1.00\nAlta"])

    ax.set_title(f"{titulo} | Corte = {threshold:.6f}", fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel(feature_1, fontsize=14, fontweight="bold")
    ax.set_ylabel(feature_2, fontsize=14, fontweight="bold")
    ax.grid(alpha=0.18)

    legenda_nao_fraude = plt.Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        markersize=8,
        markerfacecolor=AZUL_NAO_FRAUDE,
        markeredgecolor=AZUL_NAO_FRAUDE_BORDA,
        label=f"Não Fraude real ({mask_nao_fraude.sum():,})".replace(",", ".")
    )

    legenda_fraude = plt.Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        markersize=8,
        markerfacecolor=AMARELO_FRAUDE,
        markeredgecolor=PRETO_BORDA,
        label=f"Fraude real ({mask_fraude.sum():,})".replace(",", ".")
    )

    legenda_corte = plt.Line2D(
        [0],
        [0],
        linestyle="--",
        linewidth=2.4,
        color="#020617",
        label="Curva de nível da GMM no ponto de corte"
    )

    legenda_niveis = plt.Line2D(
        [0],
        [0],
        linestyle=":",
        linewidth=1.2,
        color="#475569",
        label="Curvas auxiliares: 0.25, 0.50 e 0.75"
    )

    ax.legend(
        handles=[legenda_nao_fraude, legenda_fraude, legenda_corte, legenda_niveis],
        title="Legenda",
        loc="best",
        frameon=True
    )

    plt.tight_layout()
    return fig


def gerar_secao_imagem(titulo, imagem_base64, nome_download, alt_text):

    return f"""
    <section class="plot-card">
        <div class="section-actions-only">
            <a class="download-btn link-btn" href="data:image/png;base64,{imagem_base64}" download="{html.escape(str(nome_download))}">
                Baixar PNG
            </a>
        </div>
        <img class="plot-img" src="data:image/png;base64,{imagem_base64}" alt="{html.escape(str(alt_text))}">
    </section>
    """

class IdentityScaler:
    def fit(self, X):
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            return X.to_numpy()
        return np.asarray(X)

    def fit_transform(self, X):
        return self.transform(X)


def extrair_valor_linha(linha, coluna, padrao=None):
    if coluna in linha.index and not pd.isna(linha[coluna]):
        return linha[coluna]
    return padrao


def normalizar_bool_scaler(valor):
    if pd.isna(valor):
        return True

    texto = str(valor).strip().lower()

    if texto in ["standardscaler", "standard_scaler", "sim", "true", "1", "yes"]:
        return True

    if texto in ["none", "nao", "não", "false", "0", "no"]:
        return False

    return True


def criar_tsne_2d_compat(
    perplexity,
    random_state=42,
    init="pca",
    max_iter=250,
    learning_rate="auto",
    n_jobs=-1,
    verbose=0,
    method="barnes_hut",
    angle=0.5
):

    assinatura = inspect.signature(TSNE)
    parametros = assinatura.parameters

    kwargs = {
        "n_components": 2,
        "perplexity": perplexity,
        "random_state": random_state,
        "init": init,
        "learning_rate": learning_rate,
        "method": method,
        "angle": angle,
        "verbose": verbose
    }

    if "max_iter" in parametros:
        kwargs["max_iter"] = max_iter
    else:
        kwargs["n_iter"] = max_iter

    if "n_jobs" in parametros:
        kwargs["n_jobs"] = n_jobs

    return TSNE(**kwargs)


def formatar_tempo(segundos):
    segundos = int(segundos)
    h = segundos // 3600
    m = (segundos % 3600) // 60
    s = segundos % 60

    if h > 0:
        return f"{h}h {m}min {s}s"
    if m > 0:
        return f"{m}min {s}s"
    return f"{s}s"

def processar_rank_2x2(rank, df, scores_2x2, target_name, features_tsne, pasta_latex, verbose_tsne=0):

    colunas_necessarias = [
        "Perplexity",
        "Posicao_Rank",
        "AUC_PR",
        "MCC",
        "Score_Final",
        "Melhor_Ponto_Corte"
    ]

    for coluna in colunas_necessarias:
        if coluna not in scores_2x2.columns:
            raise ValueError(f"O arquivo de scores precisa ter a coluna '{coluna}'.")

    if "Ponto_Corte_Medio" not in scores_2x2.columns:
        scores_2x2["Ponto_Corte_Medio"] = 0.5

    if rank not in scores_2x2["Posicao_Rank"].values:
        raise ValueError(
            f"Rank {rank} não encontrado. "
            f"Ranks disponíveis: 1 até {scores_2x2['Posicao_Rank'].max()}."
        )

    linha_rank = scores_2x2.loc[scores_2x2["Posicao_Rank"] == rank].iloc[0]

    feature_1 = "TSNE_1"
    feature_2 = "TSNE_2"

    perplexity = float(extrair_valor_linha(linha_rank, "Perplexity"))
    max_iter = int(float(extrair_valor_linha(linha_rank, "Max_Iter", 250)))
    init = str(extrair_valor_linha(linha_rank, "Init", "pca"))
    learning_rate = extrair_valor_linha(linha_rank, "Learning_Rate", "auto")

    if str(learning_rate).replace(".", "", 1).isdigit():
        learning_rate = float(learning_rate)

    random_state_tsne = int(float(extrair_valor_linha(linha_rank, "Random_State_TSNE", 42)))
    method_tsne = str(extrair_valor_linha(linha_rank, "Method_TSNE", "barnes_hut"))
    angle_tsne = float(extrair_valor_linha(linha_rank, "Angle_TSNE", 0.5))
    n_jobs_tsne = int(float(extrair_valor_linha(linha_rank, "N_Jobs_TSNE", -1)))

    escalar_antes_tsne = normalizar_bool_scaler(
        extrair_valor_linha(linha_rank, "Scaler_Antes_TSNE", "StandardScaler")
    )

    escalar_tsne_para_gmm = normalizar_bool_scaler(
        extrair_valor_linha(linha_rank, "Scaler_TSNE_Para_GMM", "StandardScaler")
    )

    gmm_n_components = int(float(extrair_valor_linha(linha_rank, "GMM_N_Components", 2)))
    gmm_covariance_type = str(extrair_valor_linha(linha_rank, "GMM_Covariance_Type", "full"))
    gmm_n_init = int(float(extrair_valor_linha(linha_rank, "GMM_N_Init", 3)))
    gmm_random_state = int(float(extrair_valor_linha(linha_rank, "GMM_Random_State", 42)))
    gmm_reg_covar = float(extrair_valor_linha(linha_rank, "GMM_Reg_Covar", 1e-6))

    melhor_ponto_corte = float(linha_rank["Melhor_Ponto_Corte"])
    ponto_corte_medio = float(linha_rank["Ponto_Corte_Medio"])

    auc_pr = float(linha_rank["AUC_PR"])
    mcc = float(linha_rank["MCC"])

    if "Log_Loss_Norm" in linha_rank.index and not pd.isna(linha_rank["Log_Loss_Norm"]):
        log_loss_norm = float(linha_rank["Log_Loss_Norm"])
    elif "Log_Loss" in linha_rank.index and not pd.isna(linha_rank["Log_Loss"]):
        log_loss_norm = 1 / (1 + float(linha_rank["Log_Loss"]))
    else:
        log_loss_norm = np.nan

    score_final = float(linha_rank["Score_Final"])

    if "Diferenca_Neg_Log_Veross" in linha_rank.index:
        diferenca_neg_log_veross = float(linha_rank["Diferenca_Neg_Log_Veross"])
    elif "Neg_Log_Veross_Com_Rotulo" in linha_rank.index and "Neg_Log_Veross_GMM" in linha_rank.index:
        diferenca_neg_log_veross = (
            float(linha_rank["Neg_Log_Veross_Com_Rotulo"])
            - float(linha_rank["Neg_Log_Veross_GMM"])
        )
    else:
        diferenca_neg_log_veross = np.nan

    print()
    print("=" * 80)
    print(f"PROCESSANDO RANK {rank} | t-SNE 2D | Perplexity = {perplexity:g}")
    print("=" * 80)

    tempo_inicio = time.time()

    dados = df[features_tsne + [target_name]].dropna().copy()
    X_original = dados[features_tsne].copy()
    y_real = dados[target_name].astype(int)

    if escalar_antes_tsne:
        scaler_features = StandardScaler()
        X_tsne_input = scaler_features.fit_transform(X_original)
    else:
        X_tsne_input = X_original.to_numpy()

    X_tsne_input = X_tsne_input.astype(np.float32, copy=False)

    tsne = criar_tsne_2d_compat(
        perplexity=perplexity,
        random_state=random_state_tsne,
        init=init,
        max_iter=max_iter,
        learning_rate=learning_rate,
        n_jobs=n_jobs_tsne,
        verbose=verbose_tsne,
        method=method_tsne,
        angle=angle_tsne
    )

    X_tsne = tsne.fit_transform(X_tsne_input)
    X_tsne = np.asarray(X_tsne)

    if X_tsne.shape[1] != 2:
        raise ValueError(f"O t-SNE deveria gerar 2 componentes, mas gerou shape {X_tsne.shape}.")

    temp = pd.DataFrame(
        {
            feature_1: X_tsne[:, 0],
            feature_2: X_tsne[:, 1],
            target_name: y_real.to_numpy()
        }
    )

    X = temp[[feature_1, feature_2]]

    if escalar_tsne_para_gmm:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
    else:
        scaler = IdentityScaler()
        X_scaled = scaler.fit_transform(X)

    gmm = GaussianMixture(
        n_components=gmm_n_components,
        covariance_type=gmm_covariance_type,
        random_state=gmm_random_state,
        n_init=gmm_n_init,
        reg_covar=gmm_reg_covar
    )

    gmm.fit(X_scaled)

    clusters = gmm.predict(X_scaled)
    ct = pd.crosstab(clusters, y_real)

    if 1 not in ct.columns:
        raise ValueError("A classe fraude, valor 1, não foi encontrada no target.")

    cluster_fraude = ct[1].idxmax()

    responsabilidades = gmm.predict_proba(X_scaled)
    probabilidades = responsabilidades[:, cluster_fraude]
    probabilidades = np.clip(probabilidades, 1e-15, 1 - 1e-15)

    cm_melhor = gerar_matriz_confusao(
        y_real=y_real,
        probabilidades=probabilidades,
        threshold=melhor_ponto_corte
    )

    cm_medio = gerar_matriz_confusao(
        y_real=y_real,
        probabilidades=probabilidades,
        threshold=ponto_corte_medio
    )

    valores_melhor = preparar_valores_matriz(cm_melhor)
    valores_medio = preparar_valores_matriz(cm_medio)
    valores_ideal = preparar_valores_matriz_ideal(y_real)

    tempo_rank = time.time() - tempo_inicio
    tempo_rank_formatado = formatar_tempo(tempo_rank)

    prefixo = f"rank_{rank}_tsne2d_perplexity_{sanitizar_nome(perplexity)}"
    combinacao_nome = f"t-SNE 2D | Perplexity = {perplexity:g}"

    figs = {
        "dispersao_2d": gerar_fig_dispersao_2d(temp, feature_1, feature_2, target_name),
        "boxplots": gerar_fig_boxplots_2d(temp, feature_1, feature_2, target_name),
        "spearman": gerar_fig_spearman(temp, feature_1, feature_2, target_name),
        "responsabilidades_melhor_corte": gerar_fig_responsabilidades_2d(
            df_plot=temp,
            feature_1=feature_1,
            feature_2=feature_2,
            target_name=target_name,
            scaler=scaler,
            gmm=gmm,
            cluster_fraude=cluster_fraude,
            threshold=melhor_ponto_corte,
            titulo="Responsabilidades GMM - Melhor Ponto de Corte"
        ),
        "responsabilidades_corte_medio": gerar_fig_responsabilidades_2d(
            df_plot=temp,
            feature_1=feature_1,
            feature_2=feature_2,
            target_name=target_name,
            scaler=scaler,
            gmm=gmm,
            cluster_fraude=cluster_fraude,
            threshold=ponto_corte_medio,
            titulo="Responsabilidades GMM - Ponto de Corte Médio"
        ),
        "matriz_melhor_corte": gerar_fig_matriz_confusao(
            f"Rank {rank} - {combinacao_nome} - Melhor Ponto de Corte",
            valores_melhor
        ),
        "matriz_corte_medio": gerar_fig_matriz_confusao(
            f"Rank {rank} - {combinacao_nome} - Ponto de Corte 0.5",
            valores_medio
        ),
        "matriz_ideal": gerar_fig_matriz_confusao(
            f"Rank {rank} - {combinacao_nome} - Matriz Ideal",
            valores_ideal
        )
    }

    imagens_base64 = {}

    for nome, fig in figs.items():
        nome_arquivo = f"{prefixo}_{nome}"
        salvar_figura(fig, Path(pasta_latex) / nome_arquivo)
        imagens_base64[nome] = fig_to_base64(fig)
        plt.close(fig)

    tabela_metricas_rank = pd.DataFrame([
        {
            "Rank": rank,
            "Origem": "t-SNE 2D",
            "Feature_1": feature_1,
            "Feature_2": feature_2,
            "Perplexity": perplexity,
            "N_Components_TSNE": 2,
            "Max_Iter": max_iter,
            "N_Iter_Real": extrair_valor_linha(linha_rank, "N_Iter_Real", np.nan),
            "Init": init,
            "Learning_Rate": learning_rate,
            "Random_State_TSNE": random_state_tsne,
            "Method_TSNE": method_tsne,
            "Angle_TSNE": angle_tsne,
            "N_Jobs_TSNE": n_jobs_tsne,
            "Scaler_Antes_TSNE": "StandardScaler" if escalar_antes_tsne else "None",
            "Scaler_TSNE_Para_GMM": "StandardScaler" if escalar_tsne_para_gmm else "None",
            "Melhor_Ponto_Corte": melhor_ponto_corte,
            "Ponto_Corte_Medio": ponto_corte_medio,
            "AUC_PR": auc_pr,
            "MCC": mcc,
            "Log_Loss_Norm": log_loss_norm,
            "Score_Final": score_final,
            "Diferenca_Neg_Log_Veross": diferenca_neg_log_veross,
            "GMM_N_Components": gmm_n_components,
            "GMM_Covariance_Type": gmm_covariance_type,
            "GMM_Random_State": gmm_random_state,
            "GMM_N_Init": gmm_n_init,
            "GMM_Reg_Covar": gmm_reg_covar,
            "Cluster_Fraude": int(cluster_fraude),
            "Tempo_Recriacao_TSNE_e_Relatorio": tempo_rank_formatado
        }
    ])

    tabela_matrizes_rank = pd.DataFrame([
        {
            "Rank": rank,
            "Origem": "t-SNE 2D",
            "Perplexity": perplexity,
            "Cenario": "Melhor ponto de corte",
            "Threshold": melhor_ponto_corte,
            "TN": valores_melhor["raw"]["tn"],
            "FP": valores_melhor["raw"]["fp"],
            "FN": valores_melhor["raw"]["fn"],
            "TP": valores_melhor["raw"]["tp"],
            "FN_Pct_Real_Fraude": valores_melhor["fn"]["pct"],
            "TP_Pct_Real_Fraude": valores_melhor["tp"]["pct"],
            "TN_Pct_Real_Nao_Fraude": valores_melhor["tn"]["pct"],
            "FP_Pct_Real_Nao_Fraude": valores_melhor["fp"]["pct"]
        },
        {
            "Rank": rank,
            "Origem": "t-SNE 2D",
            "Perplexity": perplexity,
            "Cenario": "Ponto de corte médio",
            "Threshold": ponto_corte_medio,
            "TN": valores_medio["raw"]["tn"],
            "FP": valores_medio["raw"]["fp"],
            "FN": valores_medio["raw"]["fn"],
            "TP": valores_medio["raw"]["tp"],
            "FN_Pct_Real_Fraude": valores_medio["fn"]["pct"],
            "TP_Pct_Real_Fraude": valores_medio["tp"]["pct"],
            "TN_Pct_Real_Nao_Fraude": valores_medio["tn"]["pct"],
            "FP_Pct_Real_Nao_Fraude": valores_medio["fp"]["pct"]
        },
        {
            "Rank": rank,
            "Origem": "t-SNE 2D",
            "Perplexity": perplexity,
            "Cenario": "Ideal",
            "Threshold": np.nan,
            "TN": valores_ideal["raw"]["tn"],
            "FP": valores_ideal["raw"]["fp"],
            "FN": valores_ideal["raw"]["fn"],
            "TP": valores_ideal["raw"]["tp"],
            "FN_Pct_Real_Fraude": valores_ideal["fn"]["pct"],
            "TP_Pct_Real_Fraude": valores_ideal["tp"]["pct"],
            "TN_Pct_Real_Nao_Fraude": valores_ideal["tn"]["pct"],
            "FP_Pct_Real_Nao_Fraude": valores_ideal["fp"]["pct"]
        }
    ])

    tabela_metricas_html = tabela_metricas_rank.copy()

    for col in tabela_metricas_html.columns:
        if pd.api.types.is_float_dtype(tabela_metricas_html[col]):
            tabela_metricas_html[col] = tabela_metricas_html[col].apply(lambda x: formatar_float_html(x, 6))

    tabela_metricas_rank_html = gerar_tabela_html(
        tabela_metricas_html,
        table_id=f"tabela_metricas_rank_{rank}"
    )

    html_metricas_rank = gerar_secao_tabela(
        titulo=f"Métricas do Rank {rank}",
        tabela_html=tabela_metricas_rank_html,
        table_id=f"tabela_metricas_rank_{rank}",
        nome_csv=f"{prefixo}_metricas.csv"
    )

    html_melhor = gerar_html_matriz(
        titulo=f"Matriz de Confusão (%) - {combinacao_nome} - Melhor Ponto de Corte",
        valores=valores_melhor,
        imagem_base64=imagens_base64["matriz_melhor_corte"],
        nome_imagem=f"{prefixo}_matriz_melhor_corte.png"
    )

    html_medio = gerar_html_matriz(
        titulo=f"Matriz de Confusão (%) - {combinacao_nome} - Ponto de Corte 0.5",
        valores=valores_medio,
        imagem_base64=imagens_base64["matriz_corte_medio"],
        nome_imagem=f"{prefixo}_matriz_corte_medio.png"
    )

    html_ideal = gerar_html_matriz(
        titulo="Matriz de Confusão Ideal (%)",
        valores=valores_ideal,
        matriz_ideal=True,
        imagem_base64=imagens_base64["matriz_ideal"],
        nome_imagem=f"{prefixo}_matriz_ideal.png"
    )

    html_rank = f"""
    <section class="rank-section" id="rank-{rank}">
        <h1>Rank {rank} - t-SNE 2D | Perplexity = {perplexity:g}</h1>

        <div class="info-box">
            <div class="info-grid">
                <div class="info-item"><div class="info-label">Rank</div><div class="info-value">{rank}</div></div>
                <div class="info-item"><div class="info-label">Features visualizadas</div><div class="info-value">TSNE_1 + TSNE_2</div></div>
                <div class="info-item"><div class="info-label">Perplexity</div><div class="info-value">{perplexity:g}</div></div>
                <div class="info-item"><div class="info-label">Max Iter</div><div class="info-value">{max_iter}</div></div>
                <div class="info-item"><div class="info-label">Init</div><div class="info-value">{html.escape(str(init))}</div></div>
                <div class="info-item"><div class="info-label">Random State t-SNE</div><div class="info-value">{random_state_tsne}</div></div>
                <div class="info-item"><div class="info-label">Melhor Ponto de Corte</div><div class="info-value">{melhor_ponto_corte:.6f}</div></div>
                <div class="info-item"><div class="info-label">Ponto de Corte Médio</div><div class="info-value">{ponto_corte_medio:.6f}</div></div>
                <div class="info-item"><div class="info-label">AUC-PR</div><div class="info-value">{auc_pr:.6f}</div></div>
                <div class="info-item"><div class="info-label">MCC</div><div class="info-value">{mcc:.6f}</div></div>
                <div class="info-item"><div class="info-label">Log Loss Norm</div><div class="info-value">{log_loss_norm:.6f}</div></div>
                <div class="info-item"><div class="info-label">Score Final</div><div class="info-value">{score_final:.6f}</div></div>
                <div class="info-item"><div class="info-label">Tempo de recriação</div><div class="info-value">{tempo_rank_formatado}</div></div>
            </div>
        </div>

        {html_metricas_rank}

        {html_melhor}

        {html_medio}

        {html_ideal}

        {gerar_secao_imagem(
            titulo=f"Dispersão 2D por Classe Real - t-SNE 2D | Perplexity = {perplexity:g}",
            imagem_base64=imagens_base64["dispersao_2d"],
            nome_download=f"{prefixo}_dispersao_2d.png",
            alt_text=f"Dispersão 2D por Classe Real - t-SNE 2D | Perplexity = {perplexity:g}"
        )}

        {gerar_secao_imagem(
            titulo=f"Boxplots dos Componentes t-SNE por Classe - Perplexity = {perplexity:g}",
            imagem_base64=imagens_base64["boxplots"],
            nome_download=f"{prefixo}_boxplots.png",
            alt_text=f"Boxplots dos Componentes t-SNE por Classe - Perplexity = {perplexity:g}"
        )}

        {gerar_secao_imagem(
            titulo=f"Correlação de Spearman entre TSNE_1, TSNE_2 e Target - Perplexity = {perplexity:g}",
            imagem_base64=imagens_base64["spearman"],
            nome_download=f"{prefixo}_spearman.png",
            alt_text=f"Correlação de Spearman entre TSNE_1, TSNE_2 e Target - Perplexity = {perplexity:g}"
        )}

        {gerar_secao_imagem(
            titulo=f"Responsabilidades Estimadas pela GMM - Melhor Ponto de Corte - t-SNE 2D | Perplexity = {perplexity:g}",
            imagem_base64=imagens_base64["responsabilidades_melhor_corte"],
            nome_download=f"{prefixo}_responsabilidades_melhor_corte.png",
            alt_text=f"Responsabilidades Estimadas pela GMM - Melhor Ponto de Corte - t-SNE 2D | Perplexity = {perplexity:g}"
        )}

        {gerar_secao_imagem(
            titulo=f"Responsabilidades Estimadas pela GMM - Ponto de Corte Médio - t-SNE 2D | Perplexity = {perplexity:g}",
            imagem_base64=imagens_base64["responsabilidades_corte_medio"],
            nome_download=f"{prefixo}_responsabilidades_corte_medio.png",
            alt_text=f"Responsabilidades Estimadas pela GMM - Ponto de Corte Médio - t-SNE 2D | Perplexity = {perplexity:g}"
        )}
    </section>
    """

    print(f"Rank {rank} finalizado em {tempo_rank_formatado}")

    return {
        "rank": rank,
        "feature_1": feature_1,
        "feature_2": feature_2,
        "perplexity": perplexity,
        "html": html_rank,
        "tabela_metricas": tabela_metricas_rank,
        "tabela_matrizes": tabela_matrizes_rank
    }

def gerar_relatorio_2x2_tsne_unificado(
    ranks=(1, 2, 3),
    arquivo_dados="creditcard.csv",
    arquivo_scores="2x2_tsne_score.csv",
    pasta_saida=".",
    nome_arquivo_html="2x2_tsne_visu_scores_tcc_faixas.html",
    pasta_latex="2x2_tsne_visu_scores_tcc_faixas",
    exportar_latex=True,
    target_name=None,
    features_tsne=None,
    verbose_tsne=0
):
    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    caminho_html = pasta_saida / nome_arquivo_html
    caminho_latex = pasta_saida / pasta_latex
    caminho_latex.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(arquivo_dados)
    scores_2x2 = pd.read_csv(arquivo_scores)

    if target_name is None:
        if "status_fraude" in df.columns:
            target_name = "status_fraude"
        elif "Class" in df.columns:
            df = df.rename(columns={"Class": "status_fraude"})
            target_name = "status_fraude"
        else:
            raise ValueError("Não encontrei a coluna target: 'status_fraude' ou 'Class'.")

    if features_tsne is None:
        features_tsne = [
            col for col in df.columns
            if col != target_name and pd.api.types.is_numeric_dtype(df[col])
        ]

    if len(features_tsne) == 0:
        raise ValueError("Nenhuma feature numérica encontrada para recriar o t-SNE.")

    print("=" * 80)
    print("RELATÓRIO VISUAL t-SNE 2D - TOP RANKS")
    print("=" * 80)
    print(f"Arquivo de dados: {arquivo_dados}")
    print(f"Arquivo de scores: {arquivo_scores}")
    print(f"HTML final: {nome_arquivo_html}")
    print(f"Pasta LaTeX/imagens: {pasta_latex}")
    print(f"Target: {target_name}")
    print(f"Features usadas para recriar o t-SNE: {len(features_tsne)}")
    print(f"Ranks solicitados: {ranks}")
    print("=" * 80)

    resultados = []

    for rank in ranks:
        resultado_rank = processar_rank_2x2(
            rank=rank,
            df=df,
            scores_2x2=scores_2x2,
            target_name=target_name,
            features_tsne=features_tsne,
            pasta_latex=caminho_latex,
            verbose_tsne=verbose_tsne
        )

        resultados.append(resultado_rank)

    tabela_metricas_total = pd.concat(
        [r["tabela_metricas"] for r in resultados],
        ignore_index=True
    )

    tabela_matrizes_total = pd.concat(
        [r["tabela_matrizes"] for r in resultados],
        ignore_index=True
    )

    if exportar_latex:
        exportar_tabelas_latex(
            pasta_latex=caminho_latex,
            tabela_metricas=tabela_metricas_total,
            tabela_matrizes=tabela_matrizes_total
        )

    tabela_metricas_total_html = tabela_metricas_total.copy()

    for col in tabela_metricas_total_html.columns:
        if pd.api.types.is_float_dtype(tabela_metricas_total_html[col]):
            tabela_metricas_total_html[col] = tabela_metricas_total_html[col].apply(lambda x: formatar_float_html(x, 6))

    tabela_matrizes_total_html = tabela_matrizes_total.copy()

    for col in tabela_matrizes_total_html.columns:
        if pd.api.types.is_float_dtype(tabela_matrizes_total_html[col]):
            tabela_matrizes_total_html[col] = tabela_matrizes_total_html[col].apply(lambda x: formatar_float_html(x, 6))

    html_tabela_metricas_total = gerar_tabela_html(
        tabela_metricas_total_html,
        table_id="tabela_metricas_ranks"
    )

    html_tabela_matrizes_total = gerar_tabela_html(
        tabela_matrizes_total_html,
        table_id="tabela_matrizes_confusao_ranks"
    )

    html_resumo_metricas = gerar_secao_tabela(
        titulo="Tabela Geral de Métricas dos Ranks",
        tabela_html=html_tabela_metricas_total,
        table_id="tabela_metricas_ranks",
        nome_csv="tabela_metricas_ranks.csv"
    )

    html_resumo_matrizes = gerar_secao_tabela(
        titulo="Tabela Geral das Matrizes de Confusão dos Ranks",
        tabela_html=html_tabela_matrizes_total,
        table_id="tabela_matrizes_confusao_ranks",
        nome_csv="tabela_matrizes_confusao_ranks.csv"
    )

    navegacao_links = "\n".join([
        f'<a href="#rank-{r["rank"]}">Rank {r["rank"]} - {html.escape(str(r["feature_1"]))} + {html.escape(str(r["feature_2"]))}</a>'
        for r in resultados
    ])

    html_ranks = "\n".join([r["html"] for r in resultados])

    html_final = f"""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <title>Relatório 2x2 t-SNE - Top Ranks</title>

        <style>
            body {{
                font-family: Arial, Helvetica, sans-serif;
                background: #f4f6f8;
                color: #020617;
                margin: 0;
                padding: 32px;
            }}

            .container {{
                max-width: 1250px;
                margin: 0 auto;
            }}

            h1 {{
                text-align: center;
                margin-bottom: 28px;
                color: #020617;
            }}

            .main-title {{
                font-size: 34px;
                margin-bottom: 14px;
            }}

            .nav-box {{
                background: #ffffff;
                border-radius: 16px;
                padding: 18px;
                margin-bottom: 28px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
                display: flex;
                justify-content: center;
                gap: 12px;
                flex-wrap: wrap;
            }}

            .nav-box a {{
                text-decoration: none;
                background: #eff6ff;
                color: #1e3a8a;
                border: 1px solid #bfdbfe;
                padding: 8px 12px;
                border-radius: 999px;
                font-weight: 800;
                font-size: 13px;
            }}

            .rank-section {{
                margin-top: 46px;
                padding-top: 12px;
                border-top: 4px solid #cbd5e1;
            }}

            .info-box {{
                background: #ffffff;
                border-radius: 16px;
                padding: 20px 24px;
                margin-bottom: 32px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
            }}

            .info-grid {{
                display: grid;
                grid-template-columns: repeat(3, 1fr);
                gap: 14px;
                margin-top: 14px;
            }}

            .info-item {{
                background: #f8fafc;
                border: 1px solid #e2e8f0;
                border-radius: 12px;
                padding: 12px 14px;
            }}

            .info-label {{
                font-size: 13px;
                font-weight: 700;
                color: #475569;
                margin-bottom: 6px;
            }}

            .info-value {{
                font-size: 18px;
                font-weight: 800;
                color: #020617;
                font-family: Consolas, Monaco, monospace;
            }}

            .matrix-card,
            .plot-card {{
                background: white;
                border-radius: 16px;
                padding: 24px;
                margin-bottom: 32px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
            }}

            .matrix-card h2,
            .plot-card h2 {{
                text-align: center;
                margin-top: 0;
                margin-bottom: 24px;
                color: #020617;
                font-size: 22px;
            }}

            .section-header {{
                display: flex;
                align-items: center;
                justify-content: center;
                gap: 14px;
                flex-wrap: wrap;
                margin-bottom: 18px;
            }}

            .section-header h2 {{
                margin: 0;
            }}

            .section-actions-only {{
                display: flex;
                justify-content: flex-end;
                margin-bottom: 12px;
            }}

            .download-btn {{
                border: 1px solid #bfdbfe;
                background: #eff6ff;
                color: #1e3a8a;
                padding: 8px 12px;
                border-radius: 10px;
                font-weight: 800;
                font-size: 13px;
                cursor: pointer;
                text-decoration: none;
                display: inline-block;
            }}

            .download-btn:hover {{
                background: #dbeafe;
            }}

            .table-wrapper {{
                overflow-x: auto;
                border: 1px solid #e2e8f0;
                border-radius: 12px;
                max-height: 580px;
                overflow-y: auto;
            }}

            .data-table {{
                width: 100%;
                border-collapse: collapse;
                font-size: 13px;
                margin-top: 0;
            }}

            .data-table th {{
                background: #0f172a;
                color: white;
                padding: 10px 8px;
                text-align: left;
                position: sticky;
                top: 0;
                z-index: 1;
            }}

            .data-table td {{
                border-bottom: 1px solid #e2e8f0;
                padding: 8px;
                color: #020617;
                white-space: nowrap;
            }}

            .data-table tr:nth-child(even) {{
                background: #f8fafc;
            }}

            .matrix-area {{
                display: flex;
                align-items: center;
                justify-content: center;
                gap: 34px;
            }}

            .matrix-wrapper {{
                display: grid;
                grid-template-columns: 180px 1fr 1fr;
                grid-template-rows: 48px 190px 190px;
                width: 950px;
            }}

            .corner {{
                background: transparent;
            }}

            .x-label {{
                display: flex;
                align-items: center;
                justify-content: center;
                font-size: 19px;
                font-weight: 700;
                color: #020617;
                border-bottom: 1px solid #e5e7eb;
            }}

            .y-label {{
                display: flex;
                align-items: center;
                justify-content: flex-end;
                padding-right: 18px;
                font-size: 19px;
                font-weight: 700;
                color: #020617;
            }}

            .cell {{
                display: flex;
                flex-direction: column;
                align-items: center;
                justify-content: center;
                min-height: 180px;
                border: 1px solid #e5e7eb;
                font-size: 20px;
                text-align: center;
                color: #020617 !important;
            }}

            .pct {{
                font-size: 30px;
                font-weight: 900;
                margin-bottom: 4px;
                color: #020617 !important;
            }}

            .count {{
                font-size: 24px;
                font-weight: 900;
                margin-bottom: 8px;
                color: #020617 !important;
            }}

            .cell-desc {{
                font-size: 13px;
                font-weight: 700;
                opacity: 1;
                color: #020617 !important;
            }}

            .q95 {{ background: #08306b; }}
            .q85 {{ background: #08519c; }}
            .q70 {{ background: #2171b5; }}
            .q50 {{ background: #6baed6; }}
            .q30 {{ background: #c6dbef; }}
            .q10 {{ background: #eff6ff; }}

            .legend {{
                position: relative;
                display: flex;
                flex-direction: column;
                align-items: center;
                min-width: 115px;
            }}

            .legend-title {{
                font-weight: 800;
                font-size: 15px;
                margin-bottom: 10px;
                color: #020617;
            }}

            .colorbar {{
                width: 30px;
                height: 310px;
                border-radius: 16px;
                background: linear-gradient(
                    to bottom,
                    #08306b 0%,
                    #08519c 18%,
                    #2171b5 36%,
                    #6baed6 58%,
                    #c6dbef 78%,
                    #eff6ff 100%
                );
                border: 1px solid #cbd5e1;
            }}

            .legend-label-top {{
                position: absolute;
                top: 43px;
                left: 78px;
                font-size: 13px;
                font-weight: 800;
                color: #020617;
            }}

            .legend-label-bottom {{
                position: absolute;
                top: 335px;
                left: 78px;
                font-size: 13px;
                font-weight: 800;
                color: #020617;
            }}

            .plot-img {{
                display: block;
                max-width: 100%;
                margin: 0 auto;
                border-radius: 12px;
                border: 1px solid #e2e8f0;
            }}

            @media (max-width: 1100px) {{
                .info-grid {{
                    grid-template-columns: repeat(2, 1fr);
                }}

                .matrix-area {{
                    flex-direction: column;
                }}

                .matrix-wrapper {{
                    width: 100%;
                    grid-template-columns: 150px 1fr 1fr;
                }}
            }}

            @media (max-width: 700px) {{
                body {{
                    padding: 16px;
                }}

                .info-grid {{
                    grid-template-columns: 1fr;
                }}

                .matrix-wrapper {{
                    grid-template-columns: 120px 1fr 1fr;
                    grid-template-rows: 48px 160px 160px;
                }}

                .pct {{
                    font-size: 22px;
                }}

                .count {{
                    font-size: 18px;
                }}

                .y-label,
                .x-label {{
                    font-size: 14px;
                }}
            }}
        </style>
    </head>

    <body>
        <div class="container">

            <h1 class="main-title">Relatório Unificado 2x2 t-SNE - Top Ranks</h1>

            <div class="nav-box">
                {navegacao_links}
            </div>

            {html_ranks}

            {html_resumo_metricas}

            {html_resumo_matrizes}

        </div>

        <script>
            function limparTextoCSV(texto) {{
                if (texto === null || texto === undefined) {{
                    return "";
                }}

                texto = String(texto).replace(/\\n/g, " ").replace(/\\s+/g, " ").trim();

                if (texto.includes(";") || texto.includes('"')) {{
                    texto = '"' + texto.replace(/"/g, '""') + '"';
                }}

                return texto;
            }}

            function baixarTabelaCSV(tableId, filename) {{
                const tabela = document.getElementById(tableId);

                if (!tabela) {{
                    alert("Tabela não encontrada: " + tableId);
                    return;
                }}

                const linhas = [];

                tabela.querySelectorAll("tr").forEach(function(row) {{
                    const celulas = Array.from(row.querySelectorAll("th, td"));
                    const linha = celulas.map(celula => limparTextoCSV(celula.innerText)).join(";");
                    linhas.push(linha);
                }});

                const csv = "\\ufeff" + linhas.join("\\n");
                const blob = new Blob([csv], {{ type: "text/csv;charset=utf-8;" }});
                const url = URL.createObjectURL(blob);

                const link = document.createElement("a");
                link.href = url;
                link.download = filename;
                document.body.appendChild(link);
                link.click();
                document.body.removeChild(link);

                URL.revokeObjectURL(url);
            }}
        </script>
    </body>
    </html>
    """

    caminho_html.write_text(html_final, encoding="utf-8")

    print("=" * 80)
    print("RELATÓRIO 2x2 t-SNE UNIFICADO GERADO COM SUCESSO")
    print("=" * 80)
    print(f"HTML salvo em: {caminho_html.resolve()}")

    if exportar_latex:
        print(f"Arquivos LaTeX/imagens salvos em: {caminho_latex.resolve()}")

    print("=" * 80)

    return {
        "caminho_html": caminho_html,
        "caminho_latex": caminho_latex,
        "tabela_metricas": tabela_metricas_total,
        "tabela_matrizes": tabela_matrizes_total
    }


resultado_2x2_tsne_visu_scores = gerar_relatorio_2x2_tsne_unificado(
    ranks=(1, 2, 3),
    arquivo_dados="creditcard.csv",
    arquivo_scores="2x2_tsne_score.csv",
    pasta_saida=".",
    nome_arquivo_html="2x2_tsne_visu_scores_tcc_faixas.html",
    pasta_latex="2x2_tsne_visu_scores_tcc_faixas",
    exportar_latex=True,
    target_name=None,
    features_tsne=None,
    verbose_tsne=0
)

resultado_2x2_tsne_visu_scores["caminho_html"]
